# Week 3.1: Giving Your Twin Hands
### Plain-English edition — no coding required

> ▶ **Do this first: click _Connect_ at the top right, now.** Colab takes 10-30 seconds to
> start a computer for you. Let it happen while you read - by the time you reach Step 1
> it will be ready, and every step will run in about a second.

> **Thought → Action → Observation → Answer**

Ask a language model what 4,817 × 23 is and it will answer confidently — and often wrongly.
It is not calculating. It is predicting what the answer *looks like*.

A **tool** fixes that. Instead of guessing, the twin stops, calls a real calculator, reads the
real result, and answers from it. A model that can do this is called an **agent**.

In this notebook you give your twin four tools and watch it decide which to reach for.

## How this notebook works
Every grey box below is a **step you run**, not code you read.

1. Some steps have **boxes to fill in** (type a word, drag a slider, tick a box).
2. Press the **▶ play button** on the left of the step.
3. Read the answer that appears underneath.

That's it. The computer writes and runs the code for you.

**Curious what it wrote?** Click **Show code** on any step to read it, then click
**Hide code** in the same place to tuck it away again.

> **Red text means something needs fixing** — read it and press play again.
> **Green means you got it right.** No red at all? You're good.

## What you will learn
- **Why tools exist** — the things a model genuinely cannot do on its own
- **A tool** — a job your twin hands over to a real piece of software
- **The toolbox** — how the twin knows what it has available
- **The loop** — thought, action, observation, answer
- **Your own tool** — a fact list only your twin knows
- **Where it breaks** — watching the twin pick the wrong tool

---
## Part 1: What a model cannot do alone

Three kinds of question defeat a model on its own: **arithmetic** it has never seen,
**anything happening now**, and **facts about you**. It will still answer all three.

In [ ]:
#@title ▶ Step 1 — Watch a twin with no tools { display-mode: "form" }
#@markdown Pick the kind of question to ask a twin that has no tools:
question_type = "a big sum" #@param ["a big sum", "what day is it today", "something only you know"]

# ---- hidden helper code: learners never need to read this ----
import sys
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

import re
def show_text(label, text, width=76):
    if label: print(label)
    for line in str(text).splitlines():
        while len(line) > width:
            cut = line.rfind(" ", 0, width)
            if cut <= 0: cut = width
            print("   " + line[:cut]); line = line[cut:].lstrip()
        print("   " + line)
def box(title, text, width=72):
    import textwrap
    print("+" + "-" * width + "+")
    print("| " + title.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
    for para in str(text).split(chr(10)):
        if not para.strip():
            print("| " + " " * (width - 2) + " |"); continue
        for line in textwrap.wrap(para, width - 4) or [""]:
            print("| " + line.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
def bar(part, whole, width=34):
    if whole <= 0: return "[" + "." * width + "]"
    f = max(0, min(width, int(round(width * part / whole))))
    return "[" + "#" * f + "." * (width - f) + "]"
def pct(part, whole):
    return (format(100.0 * part / whole, ".0f") + "%") if whole else "-"

import ast, operator, datetime
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.Pow: operator.pow, ast.Mod: operator.mod,
        ast.USub: operator.neg, ast.UAdd: operator.pos}
def safe_calc(expression):
    """Work out a sum WITHOUT python's eval, which would run any code it was given."""
    def ev(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
            return _OPS[type(node.op)](ev(node.left), ev(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
            return _OPS[type(node.op)](ev(node.operand))
        raise ValueError("only numbers and + - * / % ** are allowed")
    return ev(ast.parse(str(expression).strip(), mode="eval").body)
def tool_calculator(expression):
    try:
        return {"result": safe_calc(expression), "expression": expression}
    except Exception as e:
        return {"error": str(e)}
def tool_clock(_=None):
    now = datetime.datetime.now()
    return {"time": now.strftime("%H:%M"), "date": now.strftime("%Y-%m-%d"),
            "day": now.strftime("%A")}
RATES = {"km->miles": 0.621371, "miles->km": 1.60934, "kg->pounds": 2.20462,
         "pounds->kg": 0.453592, "celsius->fahrenheit": None}
def tool_convert(request):
    m = re.search(r"(-?\d+(?:\.\d+)?)\s*([a-zA-Z]+)", str(request))
    if not m: return {"error": "I could not find a number and a unit in that"}
    amount = float(m.group(1)); unit = m.group(2).lower()
    table = {"km": ("miles", 0.621371), "miles": ("km", 1.60934), "mile": ("km", 1.60934),
             "kg": ("pounds", 2.20462), "pounds": ("kg", 0.453592), "lb": ("kg", 0.453592)}
    if unit not in table: return {"error": "I only know km, miles, kg and pounds - not '" + unit + "'"}
    to, factor = table[unit]
    return {"result": round(amount * factor, 2), "from": str(amount) + " " + unit, "to_unit": to}
def my_facts():
    return globals().get("_FACTS", {})
def _norm(word):
    return word.lower().strip("?.,!'").rstrip("s")
def _words(text):
    return [_norm(w) for w in re.findall(r"[A-Za-z0-9]+", str(text))]
def fact_match(question):
    """Match on words, so 'pharmacy opens' still matches 'does my pharmacy open?'."""
    qw = set(_words(question))
    best, best_len = None, 0
    for key in my_facts():
        kw = [w for w in _words(key) if w]
        if kw and all(w in qw for w in kw) and len(kw) > best_len:
            best, best_len = key, len(kw)
    return best
def tool_facts(question):
    key = fact_match(question)
    if key is None:
        return {"error": "nothing in my fact list matches that"}
    return {"result": my_facts()[key], "matched": key}
TOOLBOX = {
 "calculator": (tool_calculator, "work out a sum"),
 "clock":      (tool_clock,      "say what the time, date or day is"),
 "converter":  (tool_convert,    "convert km, miles, kg and pounds"),
 "my facts":   (tool_facts,      "look something up in my own list of facts"),
}
def enabled_tools():
    return globals().get("_ENABLED", ["calculator", "clock"])
def think(question):
    """Decide which tool to reach for. Keyword matching - deliberately simple."""
    q = str(question).lower()
    on = enabled_tools()
    if "my facts" in on and fact_match(question):
        return "my facts", "this matches '" + fact_match(question) + "' in my own fact list"
    if "converter" in on and re.search(r"\d", q) and any(u in q for u in
            ("km", "mile", "kg", "pound", "lb", "convert")):
        return "converter", "there is a number next to a unit of distance or weight"
    if "clock" in on and any(w in q for w in ("time", "date", "day", "today")):
        return "clock", "the question mentions the time, the date or the day"
    if "calculator" in on and (re.search(r"\d\s*[-+*/%]\s*\d", q)
                               or any(w in q for w in ("calculate", "times", "plus", "minus"))):
        return "calculator", "there is a sum in the question"
    return None, "nothing matched, so I will answer from memory"
def extract_sum(question):
    m = re.search(r"[-+]?[\d.]+(?:\s*[-+*/%]\s*[-+]?[\d.]+)+", str(question))
    return m.group(0).strip() if m else None
def answer_from(tool_name, observation):
    if "error" in observation:
        return "I tried my " + tool_name + " but it said: " + observation["error"]
    if tool_name == "clock":
        return "It is " + observation["time"] + " on " + observation["day"] + ", " + observation["date"] + "."
    if tool_name == "converter":
        return observation["from"] + " is " + str(observation["result"]) + " " + observation["to_unit"] + "."
    return "The answer is " + str(observation["result"]) + "."
def run_agent(question, verbose=True):
    """Thought -> Action -> Observation -> Answer."""
    tool_name, why = think(question)
    if verbose:
        print("THOUGHT      : " + why)
    if tool_name is None:
        if verbose:
            print("ACTION       : none - answering from memory")
            print("OBSERVATION  : (nothing was looked up)")
        return None, "I do not have a tool for that, so I would have to guess."
    arg = extract_sum(question) if tool_name == "calculator" else question
    if tool_name == "calculator" and arg is None:
        arg = question
    if verbose:
        print("ACTION       : use the " + tool_name + " with  " + str(arg)[:50])
    observation = TOOLBOX[tool_name][0](arg)
    if verbose:
        print("OBSERVATION  : " + str(observation))
    return tool_name, answer_from(tool_name, observation)

DEMO = {
 "a big sum": ("What is 4817 * 23?", "I think that is about 110,000.",
               str(safe_calc("4817 * 23"))),
 "what day is it today": ("What day is it today?",
               "I believe it is Tuesday.", tool_clock()["day"]),
 "something only you know": ("What time does my pharmacy open?",
               "Most pharmacies open around 9am.", "7am - but only your notes know that"),
}
q, guess, truth = DEMO[question_type]
print("Question: " + q)
print("")
box("A twin with NO tools says", guess)
print("")
problem("That answer was produced by predicting words, not by checking anything.")
print("")
print("The real answer: " + str(truth))
print("")
print("Notice the guess SOUNDS fine. That is the danger - there is nothing in the")
print("wording to tell you it was never checked.")
print("")
note("A tool replaces the guess with a lookup. That is the whole idea.")

---
## Part 2: A tool is just a job handed over

A tool is a small piece of software with a **name**, a **description** of when to use it, and
something it actually does. Here is the simplest one: a calculator.

In [1]:
#@title ▶ Step 2 — Try the calculator tool on its own { display-mode: "form" }
#@markdown Type a sum. Try one no model could memorise.
sum_to_do = "4817 * 23" #@param {type:"string"}

# ---- hidden helper code: learners never need to read this ----
import sys
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

import re
def show_text(label, text, width=76):
    if label: print(label)
    for line in str(text).splitlines():
        while len(line) > width:
            cut = line.rfind(" ", 0, width)
            if cut <= 0: cut = width
            print("   " + line[:cut]); line = line[cut:].lstrip()
        print("   " + line)
def box(title, text, width=72):
    import textwrap
    print("+" + "-" * width + "+")
    print("| " + title.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
    for para in str(text).split(chr(10)):
        if not para.strip():
            print("| " + " " * (width - 2) + " |"); continue
        for line in textwrap.wrap(para, width - 4) or [""]:
            print("| " + line.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
def bar(part, whole, width=34):
    if whole <= 0: return "[" + "." * width + "]"
    f = max(0, min(width, int(round(width * part / whole))))
    return "[" + "#" * f + "." * (width - f) + "]"
def pct(part, whole):
    return (format(100.0 * part / whole, ".0f") + "%") if whole else "-"

import ast, operator, datetime
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.Pow: operator.pow, ast.Mod: operator.mod,
        ast.USub: operator.neg, ast.UAdd: operator.pos}
def safe_calc(expression):
    """Work out a sum WITHOUT python's eval, which would run any code it was given."""
    def ev(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
            return _OPS[type(node.op)](ev(node.left), ev(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
            return _OPS[type(node.op)](ev(node.operand))
        raise ValueError("only numbers and + - * / % ** are allowed")
    return ev(ast.parse(str(expression).strip(), mode="eval").body)
def tool_calculator(expression):
    try:
        return {"result": safe_calc(expression), "expression": expression}
    except Exception as e:
        return {"error": str(e)}
def tool_clock(_=None):
    now = datetime.datetime.now()
    return {"time": now.strftime("%H:%M"), "date": now.strftime("%Y-%m-%d"),
            "day": now.strftime("%A")}
RATES = {"km->miles": 0.621371, "miles->km": 1.60934, "kg->pounds": 2.20462,
         "pounds->kg": 0.453592, "celsius->fahrenheit": None}
def tool_convert(request):
    m = re.search(r"(-?\d+(?:\.\d+)?)\s*([a-zA-Z]+)", str(request))
    if not m: return {"error": "I could not find a number and a unit in that"}
    amount = float(m.group(1)); unit = m.group(2).lower()
    table = {"km": ("miles", 0.621371), "miles": ("km", 1.60934), "mile": ("km", 1.60934),
             "kg": ("pounds", 2.20462), "pounds": ("kg", 0.453592), "lb": ("kg", 0.453592)}
    if unit not in table: return {"error": "I only know km, miles, kg and pounds - not '" + unit + "'"}
    to, factor = table[unit]
    return {"result": round(amount * factor, 2), "from": str(amount) + " " + unit, "to_unit": to}
def my_facts():
    return globals().get("_FACTS", {})
def _norm(word):
    return word.lower().strip("?.,!'").rstrip("s")
def _words(text):
    return [_norm(w) for w in re.findall(r"[A-Za-z0-9]+", str(text))]
def fact_match(question):
    """Match on words, so 'pharmacy opens' still matches 'does my pharmacy open?'."""
    qw = set(_words(question))
    best, best_len = None, 0
    for key in my_facts():
        kw = [w for w in _words(key) if w]
        if kw and all(w in qw for w in kw) and len(kw) > best_len:
            best, best_len = key, len(kw)
    return best
def tool_facts(question):
    key = fact_match(question)
    if key is None:
        return {"error": "nothing in my fact list matches that"}
    return {"result": my_facts()[key], "matched": key}
TOOLBOX = {
 "calculator": (tool_calculator, "work out a sum"),
 "clock":      (tool_clock,      "say what the time, date or day is"),
 "converter":  (tool_convert,    "convert km, miles, kg and pounds"),
 "my facts":   (tool_facts,      "look something up in my own list of facts"),
}
def enabled_tools():
    return globals().get("_ENABLED", ["calculator", "clock"])
def think(question):
    """Decide which tool to reach for. Keyword matching - deliberately simple."""
    q = str(question).lower()
    on = enabled_tools()
    if "my facts" in on and fact_match(question):
        return "my facts", "this matches '" + fact_match(question) + "' in my own fact list"
    if "converter" in on and re.search(r"\d", q) and any(u in q for u in
            ("km", "mile", "kg", "pound", "lb", "convert")):
        return "converter", "there is a number next to a unit of distance or weight"
    if "clock" in on and any(w in q for w in ("time", "date", "day", "today")):
        return "clock", "the question mentions the time, the date or the day"
    if "calculator" in on and (re.search(r"\d\s*[-+*/%]\s*\d", q)
                               or any(w in q for w in ("calculate", "times", "plus", "minus"))):
        return "calculator", "there is a sum in the question"
    return None, "nothing matched, so I will answer from memory"
def extract_sum(question):
    m = re.search(r"[-+]?[\d.]+(?:\s*[-+*/%]\s*[-+]?[\d.]+)+", str(question))
    return m.group(0).strip() if m else None
def answer_from(tool_name, observation):
    if "error" in observation:
        return "I tried my " + tool_name + " but it said: " + observation["error"]
    if tool_name == "clock":
        return "It is " + observation["time"] + " on " + observation["day"] + ", " + observation["date"] + "."
    if tool_name == "converter":
        return observation["from"] + " is " + str(observation["result"]) + " " + observation["to_unit"] + "."
    return "The answer is " + str(observation["result"]) + "."
def run_agent(question, verbose=True):
    """Thought -> Action -> Observation -> Answer."""
    tool_name, why = think(question)
    if verbose:
        print("THOUGHT      : " + why)
    if tool_name is None:
        if verbose:
            print("ACTION       : none - answering from memory")
            print("OBSERVATION  : (nothing was looked up)")
        return None, "I do not have a tool for that, so I would have to guess."
    arg = extract_sum(question) if tool_name == "calculator" else question
    if tool_name == "calculator" and arg is None:
        arg = question
    if verbose:
        print("ACTION       : use the " + tool_name + " with  " + str(arg)[:50])
    observation = TOOLBOX[tool_name][0](arg)
    if verbose:
        print("OBSERVATION  : " + str(observation))
    return tool_name, answer_from(tool_name, observation)

print("TOOL        : calculator")
print("DESCRIPTION : work out a sum")
print("GIVEN       : " + sum_to_do)
print("")
out = tool_calculator(sum_to_do)
if "error" in out:
    problem("The tool refused: " + out["error"])
    note("It only accepts numbers and + - * / % ** - try something like 4817 * 23.")
else:
    success("RETURNED    : " + str(out["result"]))
    print("")
    print("No guessing, no 'about'. The sum was actually done.")
    print("")
    print("Worth knowing: this tool refuses anything that is not arithmetic. A tool")
    print("that ran whatever it was handed would let a stranger run code on your")
    print("computer, just by typing it into your twin.")

TOOL        : calculator
DESCRIPTION : work out a sum
GIVEN       : 4817 * 23




No guessing, no 'about'. The sum was actually done.

Worth knowing: this tool refuses anything that is not arithmetic. A tool
that ran whatever it was handed would let a stranger run code on your
computer, just by typing it into your twin.


---
## Part 3: The toolbox

Your twin can hold several tools. It needs a **list** of them, with a description of each,
so it can work out which one fits a question. Switch tools on and off below.

In [3]:
#@title ▶ Step 3 — Choose what goes in the toolbox { display-mode: "form" }
calculator = True #@param {type:"boolean"}
clock = True #@param {type:"boolean"}
converter = True #@param {type:"boolean"}
my_facts = True #@param {type:"boolean"}
#@markdown Your own facts, for the "my facts" tool (one per line, as `thing = answer`):
my_fact_list = "pharmacy opens = 7am; best 5km = 23 minutes; exam = June" #@param {type:"string"}

# ---- hidden helper code: learners never need to read this ----
import sys
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

import re
def show_text(label, text, width=76):
    if label: print(label)
    for line in str(text).splitlines():
        while len(line) > width:
            cut = line.rfind(" ", 0, width)
            if cut <= 0: cut = width
            print("   " + line[:cut]); line = line[cut:].lstrip()
        print("   " + line)
def box(title, text, width=72):
    import textwrap
    print("+" + "-" * width + "+")
    print("| " + title.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
    for para in str(text).split(chr(10)):
        if not para.strip():
            print("| " + " " * (width - 2) + " |"); continue
        for line in textwrap.wrap(para, width - 4) or [""]:
            print("| " + line.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
def bar(part, whole, width=34):
    if whole <= 0: return "[" + "." * width + "]"
    f = max(0, min(width, int(round(width * part / whole))))
    return "[" + "#" * f + "." * (width - f) + "]"
def pct(part, whole):
    return (format(100.0 * part / whole, ".0f") + "%") if whole else "-"

import ast, operator, datetime
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.Pow: operator.pow, ast.Mod: operator.mod,
        ast.USub: operator.neg, ast.UAdd: operator.pos}
def safe_calc(expression):
    """Work out a sum WITHOUT python's eval, which would run any code it was given."""
    def ev(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
            return _OPS[type(node.op)](ev(node.left), ev(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
            return _OPS[type(node.op)](ev(node.operand))
        raise ValueError("only numbers and + - * / % ** are allowed")
    return ev(ast.parse(str(expression).strip(), mode="eval").body)
def tool_calculator(expression):
    try:
        return {"result": safe_calc(expression), "expression": expression}
    except Exception as e:
        return {"error": str(e)}
def tool_clock(_=None):
    now = datetime.datetime.now()
    return {"time": now.strftime("%H:%M"), "date": now.strftime("%Y-%m-%d"),
            "day": now.strftime("%A")}
RATES = {"km->miles": 0.621371, "miles->km": 1.60934, "kg->pounds": 2.20462,
         "pounds->kg": 0.453592, "celsius->fahrenheit": None}
def tool_convert(request):
    m = re.search(r"(-?\d+(?:\.\d+)?)\s*([a-zA-Z]+)", str(request))
    if not m: return {"error": "I could not find a number and a unit in that"}
    amount = float(m.group(1)); unit = m.group(2).lower()
    table = {"km": ("miles", 0.621371), "miles": ("km", 1.60934), "mile": ("km", 1.60934),
             "kg": ("pounds", 2.20462), "pounds": ("kg", 0.453592), "lb": ("kg", 0.453592)}
    if unit not in table: return {"error": "I only know km, miles, kg and pounds - not '" + unit + "'"}
    to, factor = table[unit]
    return {"result": round(amount * factor, 2), "from": str(amount) + " " + unit, "to_unit": to}
def my_facts():
    return globals().get("_FACTS", {})
def _norm(word):
    return word.lower().strip("?.,!'").rstrip("s")
def _words(text):
    return [_norm(w) for w in re.findall(r"[A-Za-z0-9]+", str(text))]
def fact_match(question):
    """Match on words, so 'pharmacy opens' still matches 'does my pharmacy open?'."""
    qw = set(_words(question))
    best, best_len = None, 0
    for key in my_facts():
        kw = [w for w in _words(key) if w]
        if kw and all(w in qw for w in kw) and len(kw) > best_len:
            best, best_len = key, len(kw)
    return best
def tool_facts(question):
    key = fact_match(question)
    if key is None:
        return {"error": "nothing in my fact list matches that"}
    return {"result": my_facts()[key], "matched": key}
TOOLBOX = {
 "calculator": (tool_calculator, "work out a sum"),
 "clock":      (tool_clock,      "say what the time, date or day is"),
 "converter":  (tool_convert,    "convert km, miles, kg and pounds"),
 "my facts":   (tool_facts,      "look something up in my own list of facts"),
}
def enabled_tools():
    return globals().get("_ENABLED", ["calculator", "clock"])
def think(question):
    """Decide which tool to reach for. Keyword matching - deliberately simple."""
    q = str(question).lower()
    on = enabled_tools()
    if "my facts" in on and fact_match(question):
        return "my facts", "this matches '" + fact_match(question) + "' in my own fact list"
    if "converter" in on and re.search(r"\d", q) and any(u in q for u in
            ("km", "mile", "kg", "pound", "lb", "convert")):
        return "converter", "there is a number next to a unit of distance or weight"
    if "clock" in on and any(w in q for w in ("time", "date", "day", "today")):
        return "clock", "the question mentions the time, the date or the day"
    if "calculator" in on and (re.search(r"\d\s*[-+*/%]\s*\d", q)
                               or any(w in q for w in ("calculate", "times", "plus", "minus"))):
        return "calculator", "there is a sum in the question"
    return None, "nothing matched, so I will answer from memory"
def extract_sum(question):
    m = re.search(r"[-+]?[\d.]+(?:\s*[-+*/%]\s*[-+]?[\d.]+)+", str(question))
    return m.group(0).strip() if m else None
def answer_from(tool_name, observation):
    if "error" in observation:
        return "I tried my " + tool_name + " but it said: " + observation["error"]
    if tool_name == "clock":
        return "It is " + observation["time"] + " on " + observation["day"] + ", " + observation["date"] + "."
    if tool_name == "converter":
        return observation["from"] + " is " + str(observation["result"]) + " " + observation["to_unit"] + "."
    return "The answer is " + str(observation["result"]) + "."
def run_agent(question, verbose=True):
    """Thought -> Action -> Observation -> Answer."""
    tool_name, why = think(question)
    if verbose:
        print("THOUGHT      : " + why)
    if tool_name is None:
        if verbose:
            print("ACTION       : none - answering from memory")
            print("OBSERVATION  : (nothing was looked up)")
        return None, "I do not have a tool for that, so I would have to guess."
    arg = extract_sum(question) if tool_name == "calculator" else question
    if tool_name == "calculator" and arg is None:
        arg = question
    if verbose:
        print("ACTION       : use the " + tool_name + " with  " + str(arg)[:50])
    observation = TOOLBOX[tool_name][0](arg)
    if verbose:
        print("OBSERVATION  : " + str(observation))
    return tool_name, answer_from(tool_name, observation)

on = [name for name, flag in [("calculator", calculator), ("clock", clock),
                              ("converter", converter), ("my facts", my_facts)] if flag]
globals()["_ENABLED"] = on
facts = {}
for pair in my_fact_list.split(";"):
    if "=" in pair:
        k, v = pair.split("=", 1)
        if k.strip(): facts[k.strip()] = v.strip()
globals()["_FACTS"] = facts
if not on:
    problem("Every tool is switched off, so your twin can only guess.")
    note("Tick at least one and press play again.")
else:
    print("Your twin's toolbox:")
    print("")
    for name in on:
        print("   " + name.ljust(12) + " - " + TOOLBOX[name][1])
    print("")
    if "my facts" in on:
        if facts:
            print("Your own facts (" + str(len(facts)) + "):")
            for k, v in facts.items():
                print("   " + k.ljust(16) + " -> " + v)
            print("")
        else:
            problem("The 'my facts' tool is on, but your fact list is empty or malformed.")
            note("Write it like this:  pharmacy opens = 7am; best 5km = 23 minutes")
    success(str(len(on)) + " tool(s) ready. Step 4 lets the twin choose between them.")
    print("")
    print("This list is exactly what gets described to the model. If a tool is not")
    print("on the list, the twin does not know it exists.")

Your twin's toolbox:

   calculator   - work out a sum
   clock        - say what the time, date or day is
   converter    - convert km, miles, kg and pounds
   my facts     - look something up in my own list of facts

Your own facts (3):
   pharmacy opens   -> 7am
   best 5km         -> 23 minutes
   exam             -> June




This list is exactly what gets described to the model. If a tool is not
on the list, the twin does not know it exists.


---
## Part 4: The loop

This is the pattern behind every AI agent, and it has four moves:

| Move | What happens |
|---|---|
| **Thought** | Which tool, if any, fits this question? |
| **Action** | Call that tool with the right input |
| **Observation** | Read what came back |
| **Answer** | Reply using what came back, not from memory |

In [4]:
#@title ▶ Step 4 — Watch my twin think, act and answer { display-mode: "form" }
question = "What is 4817 * 23?" #@param {type:"string"}
#@markdown Which tools does it have? (same list as Step 3)
calculator = True #@param {type:"boolean"}
clock = True #@param {type:"boolean"}
converter = True #@param {type:"boolean"}
my_facts = True #@param {type:"boolean"}
my_fact_list = "pharmacy opens = 7am; best 5km = 23 minutes; exam = June" #@param {type:"string"}

# ---- hidden helper code: learners never need to read this ----
import sys
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

import re
def show_text(label, text, width=76):
    if label: print(label)
    for line in str(text).splitlines():
        while len(line) > width:
            cut = line.rfind(" ", 0, width)
            if cut <= 0: cut = width
            print("   " + line[:cut]); line = line[cut:].lstrip()
        print("   " + line)
def box(title, text, width=72):
    import textwrap
    print("+" + "-" * width + "+")
    print("| " + title.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
    for para in str(text).split(chr(10)):
        if not para.strip():
            print("| " + " " * (width - 2) + " |"); continue
        for line in textwrap.wrap(para, width - 4) or [""]:
            print("| " + line.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
def bar(part, whole, width=34):
    if whole <= 0: return "[" + "." * width + "]"
    f = max(0, min(width, int(round(width * part / whole))))
    return "[" + "#" * f + "." * (width - f) + "]"
def pct(part, whole):
    return (format(100.0 * part / whole, ".0f") + "%") if whole else "-"

import ast, operator, datetime
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.Pow: operator.pow, ast.Mod: operator.mod,
        ast.USub: operator.neg, ast.UAdd: operator.pos}
def safe_calc(expression):
    """Work out a sum WITHOUT python's eval, which would run any code it was given."""
    def ev(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
            return _OPS[type(node.op)](ev(node.left), ev(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
            return _OPS[type(node.op)](ev(node.operand))
        raise ValueError("only numbers and + - * / % ** are allowed")
    return ev(ast.parse(str(expression).strip(), mode="eval").body)
def tool_calculator(expression):
    try:
        return {"result": safe_calc(expression), "expression": expression}
    except Exception as e:
        return {"error": str(e)}
def tool_clock(_=None):
    now = datetime.datetime.now()
    return {"time": now.strftime("%H:%M"), "date": now.strftime("%Y-%m-%d"),
            "day": now.strftime("%A")}
RATES = {"km->miles": 0.621371, "miles->km": 1.60934, "kg->pounds": 2.20462,
         "pounds->kg": 0.453592, "celsius->fahrenheit": None}
def tool_convert(request):
    m = re.search(r"(-?\d+(?:\.\d+)?)\s*([a-zA-Z]+)", str(request))
    if not m: return {"error": "I could not find a number and a unit in that"}
    amount = float(m.group(1)); unit = m.group(2).lower()
    table = {"km": ("miles", 0.621371), "miles": ("km", 1.60934), "mile": ("km", 1.60934),
             "kg": ("pounds", 2.20462), "pounds": ("kg", 0.453592), "lb": ("kg", 0.453592)}
    if unit not in table: return {"error": "I only know km, miles, kg and pounds - not '" + unit + "'"}
    to, factor = table[unit]
    return {"result": round(amount * factor, 2), "from": str(amount) + " " + unit, "to_unit": to}
def my_facts():
    return globals().get("_FACTS", {})
def _norm(word):
    return word.lower().strip("?.,!'").rstrip("s")
def _words(text):
    return [_norm(w) for w in re.findall(r"[A-Za-z0-9]+", str(text))]
def fact_match(question):
    """Match on words, so 'pharmacy opens' still matches 'does my pharmacy open?'."""
    qw = set(_words(question))
    best, best_len = None, 0
    for key in my_facts():
        kw = [w for w in _words(key) if w]
        if kw and all(w in qw for w in kw) and len(kw) > best_len:
            best, best_len = key, len(kw)
    return best
def tool_facts(question):
    key = fact_match(question)
    if key is None:
        return {"error": "nothing in my fact list matches that"}
    return {"result": my_facts()[key], "matched": key}
TOOLBOX = {
 "calculator": (tool_calculator, "work out a sum"),
 "clock":      (tool_clock,      "say what the time, date or day is"),
 "converter":  (tool_convert,    "convert km, miles, kg and pounds"),
 "my facts":   (tool_facts,      "look something up in my own list of facts"),
}
def enabled_tools():
    return globals().get("_ENABLED", ["calculator", "clock"])
def think(question):
    """Decide which tool to reach for. Keyword matching - deliberately simple."""
    q = str(question).lower()
    on = enabled_tools()
    if "my facts" in on and fact_match(question):
        return "my facts", "this matches '" + fact_match(question) + "' in my own fact list"
    if "converter" in on and re.search(r"\d", q) and any(u in q for u in
            ("km", "mile", "kg", "pound", "lb", "convert")):
        return "converter", "there is a number next to a unit of distance or weight"
    if "clock" in on and any(w in q for w in ("time", "date", "day", "today")):
        return "clock", "the question mentions the time, the date or the day"
    if "calculator" in on and (re.search(r"\d\s*[-+*/%]\s*\d", q)
                               or any(w in q for w in ("calculate", "times", "plus", "minus"))):
        return "calculator", "there is a sum in the question"
    return None, "nothing matched, so I will answer from memory"
def extract_sum(question):
    m = re.search(r"[-+]?[\d.]+(?:\s*[-+*/%]\s*[-+]?[\d.]+)+", str(question))
    return m.group(0).strip() if m else None
def answer_from(tool_name, observation):
    if "error" in observation:
        return "I tried my " + tool_name + " but it said: " + observation["error"]
    if tool_name == "clock":
        return "It is " + observation["time"] + " on " + observation["day"] + ", " + observation["date"] + "."
    if tool_name == "converter":
        return observation["from"] + " is " + str(observation["result"]) + " " + observation["to_unit"] + "."
    return "The answer is " + str(observation["result"]) + "."
def run_agent(question, verbose=True):
    """Thought -> Action -> Observation -> Answer."""
    tool_name, why = think(question)
    if verbose:
        print("THOUGHT      : " + why)
    if tool_name is None:
        if verbose:
            print("ACTION       : none - answering from memory")
            print("OBSERVATION  : (nothing was looked up)")
        return None, "I do not have a tool for that, so I would have to guess."
    arg = extract_sum(question) if tool_name == "calculator" else question
    if tool_name == "calculator" and arg is None:
        arg = question
    if verbose:
        print("ACTION       : use the " + tool_name + " with  " + str(arg)[:50])
    observation = TOOLBOX[tool_name][0](arg)
    if verbose:
        print("OBSERVATION  : " + str(observation))
    return tool_name, answer_from(tool_name, observation)

globals()["_ENABLED"] = [n_ for n_, f in [("calculator", calculator), ("clock", clock),
                         ("converter", converter), ("my facts", my_facts)] if f]
facts = {}
for pair in my_fact_list.split(";"):
    if "=" in pair:
        k, v = pair.split("=", 1)
        if k.strip(): facts[k.strip()] = v.strip()
globals()["_FACTS"] = facts
if not question.strip():
    problem("Please type a question, then press play again.")
else:
    print("QUESTION     : " + question)
    print("")
    used, reply = run_agent(question)
    print("")
    box("ANSWER", reply)
    print("")
    if used:
        success("Your twin used its " + used + " tool instead of guessing.")
    else:
        problem("No tool matched, so the twin had nothing to look up.")
        note("Either switch on a tool that fits, or try: 'what is 88 / 4', "
             "'what day is it', '10 km in miles', or something from your fact list.")

QUESTION     : What is 4817 * 23?

THOUGHT      : there is a sum in the question
ACTION       : use the calculator with  4817 * 23
OBSERVATION  : {'result': 110791, 'expression': '4817 * 23'}

+------------------------------------------------------------------------+
| ANSWER                                                                 |
+------------------------------------------------------------------------+
| The answer is 110791.                                                  |
+------------------------------------------------------------------------+



---
## Part 5: A tool only your twin has

The three built-in tools are generic. The one that makes a twin *yours* is the fact list —
things no model anywhere could know.

In [5]:
#@title ▶ Step 5 — Teach my twin something only I know { display-mode: "form" }
#@markdown Your own facts, written as `thing = answer`, separated by semicolons:
my_fact_list = "pharmacy opens = 7am; my bus = number 14; landlord = Mr Otieno; best 5km = 23 minutes" #@param {type:"string"}
#@markdown Now ask about one of them. Use the same words as the left-hand side.
question = "What time does my pharmacy open?" #@param {type:"string"}

# ---- hidden helper code: learners never need to read this ----
import sys
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

import re
def show_text(label, text, width=76):
    if label: print(label)
    for line in str(text).splitlines():
        while len(line) > width:
            cut = line.rfind(" ", 0, width)
            if cut <= 0: cut = width
            print("   " + line[:cut]); line = line[cut:].lstrip()
        print("   " + line)
def box(title, text, width=72):
    import textwrap
    print("+" + "-" * width + "+")
    print("| " + title.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
    for para in str(text).split(chr(10)):
        if not para.strip():
            print("| " + " " * (width - 2) + " |"); continue
        for line in textwrap.wrap(para, width - 4) or [""]:
            print("| " + line.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
def bar(part, whole, width=34):
    if whole <= 0: return "[" + "." * width + "]"
    f = max(0, min(width, int(round(width * part / whole))))
    return "[" + "#" * f + "." * (width - f) + "]"
def pct(part, whole):
    return (format(100.0 * part / whole, ".0f") + "%") if whole else "-"

import ast, operator, datetime
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.Pow: operator.pow, ast.Mod: operator.mod,
        ast.USub: operator.neg, ast.UAdd: operator.pos}
def safe_calc(expression):
    """Work out a sum WITHOUT python's eval, which would run any code it was given."""
    def ev(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
            return _OPS[type(node.op)](ev(node.left), ev(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
            return _OPS[type(node.op)](ev(node.operand))
        raise ValueError("only numbers and + - * / % ** are allowed")
    return ev(ast.parse(str(expression).strip(), mode="eval").body)
def tool_calculator(expression):
    try:
        return {"result": safe_calc(expression), "expression": expression}
    except Exception as e:
        return {"error": str(e)}
def tool_clock(_=None):
    now = datetime.datetime.now()
    return {"time": now.strftime("%H:%M"), "date": now.strftime("%Y-%m-%d"),
            "day": now.strftime("%A")}
RATES = {"km->miles": 0.621371, "miles->km": 1.60934, "kg->pounds": 2.20462,
         "pounds->kg": 0.453592, "celsius->fahrenheit": None}
def tool_convert(request):
    m = re.search(r"(-?\d+(?:\.\d+)?)\s*([a-zA-Z]+)", str(request))
    if not m: return {"error": "I could not find a number and a unit in that"}
    amount = float(m.group(1)); unit = m.group(2).lower()
    table = {"km": ("miles", 0.621371), "miles": ("km", 1.60934), "mile": ("km", 1.60934),
             "kg": ("pounds", 2.20462), "pounds": ("kg", 0.453592), "lb": ("kg", 0.453592)}
    if unit not in table: return {"error": "I only know km, miles, kg and pounds - not '" + unit + "'"}
    to, factor = table[unit]
    return {"result": round(amount * factor, 2), "from": str(amount) + " " + unit, "to_unit": to}
def my_facts():
    return globals().get("_FACTS", {})
def _norm(word):
    return word.lower().strip("?.,!'").rstrip("s")
def _words(text):
    return [_norm(w) for w in re.findall(r"[A-Za-z0-9]+", str(text))]
def fact_match(question):
    """Match on words, so 'pharmacy opens' still matches 'does my pharmacy open?'."""
    qw = set(_words(question))
    best, best_len = None, 0
    for key in my_facts():
        kw = [w for w in _words(key) if w]
        if kw and all(w in qw for w in kw) and len(kw) > best_len:
            best, best_len = key, len(kw)
    return best
def tool_facts(question):
    key = fact_match(question)
    if key is None:
        return {"error": "nothing in my fact list matches that"}
    return {"result": my_facts()[key], "matched": key}
TOOLBOX = {
 "calculator": (tool_calculator, "work out a sum"),
 "clock":      (tool_clock,      "say what the time, date or day is"),
 "converter":  (tool_convert,    "convert km, miles, kg and pounds"),
 "my facts":   (tool_facts,      "look something up in my own list of facts"),
}
def enabled_tools():
    return globals().get("_ENABLED", ["calculator", "clock"])
def think(question):
    """Decide which tool to reach for. Keyword matching - deliberately simple."""
    q = str(question).lower()
    on = enabled_tools()
    if "my facts" in on and fact_match(question):
        return "my facts", "this matches '" + fact_match(question) + "' in my own fact list"
    if "converter" in on and re.search(r"\d", q) and any(u in q for u in
            ("km", "mile", "kg", "pound", "lb", "convert")):
        return "converter", "there is a number next to a unit of distance or weight"
    if "clock" in on and any(w in q for w in ("time", "date", "day", "today")):
        return "clock", "the question mentions the time, the date or the day"
    if "calculator" in on and (re.search(r"\d\s*[-+*/%]\s*\d", q)
                               or any(w in q for w in ("calculate", "times", "plus", "minus"))):
        return "calculator", "there is a sum in the question"
    return None, "nothing matched, so I will answer from memory"
def extract_sum(question):
    m = re.search(r"[-+]?[\d.]+(?:\s*[-+*/%]\s*[-+]?[\d.]+)+", str(question))
    return m.group(0).strip() if m else None
def answer_from(tool_name, observation):
    if "error" in observation:
        return "I tried my " + tool_name + " but it said: " + observation["error"]
    if tool_name == "clock":
        return "It is " + observation["time"] + " on " + observation["day"] + ", " + observation["date"] + "."
    if tool_name == "converter":
        return observation["from"] + " is " + str(observation["result"]) + " " + observation["to_unit"] + "."
    return "The answer is " + str(observation["result"]) + "."
def run_agent(question, verbose=True):
    """Thought -> Action -> Observation -> Answer."""
    tool_name, why = think(question)
    if verbose:
        print("THOUGHT      : " + why)
    if tool_name is None:
        if verbose:
            print("ACTION       : none - answering from memory")
            print("OBSERVATION  : (nothing was looked up)")
        return None, "I do not have a tool for that, so I would have to guess."
    arg = extract_sum(question) if tool_name == "calculator" else question
    if tool_name == "calculator" and arg is None:
        arg = question
    if verbose:
        print("ACTION       : use the " + tool_name + " with  " + str(arg)[:50])
    observation = TOOLBOX[tool_name][0](arg)
    if verbose:
        print("OBSERVATION  : " + str(observation))
    return tool_name, answer_from(tool_name, observation)

facts = {}
for pair in my_fact_list.split(";"):
    if "=" in pair:
        k, v = pair.split("=", 1)
        if k.strip(): facts[k.strip()] = v.strip()
globals()["_FACTS"] = facts
globals()["_ENABLED"] = ["my facts", "calculator", "clock", "converter"]
if not facts:
    problem("I could not read any facts from that list.")
    note("Write it like this:  pharmacy opens = 7am; my bus = number 14")
else:
    print("Your twin now knows " + str(len(facts)) + " private fact(s):")
    for k, v in facts.items():
        print("   " + k.ljust(18) + " -> " + v)
    print("")
    print("QUESTION     : " + question)
    print("")
    used, reply = run_agent(question)
    print("")
    box("ANSWER", reply)
    print("")
    if used == "my facts":
        success("Looked up in YOUR list. No model on earth could have known that.")
    else:
        problem("Your fact list was not used" + (" - the " + used + " tool answered instead." if used else "."))
        note("The question must contain every word on the left of the '='. "
             "Ask about '" + list(facts)[0] + "' and press play again.")

Your twin now knows 4 private fact(s):
   pharmacy opens     -> 7am
   my bus             -> number 14
   landlord           -> Mr Otieno
   best 5km           -> 23 minutes

QUESTION     : What time does my pharmacy open?

THOUGHT      : this matches 'pharmacy opens' in my own fact list
ACTION       : use the my facts with  What time does my pharmacy open?
OBSERVATION  : {'result': '7am', 'matched': 'pharmacy opens'}

+------------------------------------------------------------------------+
| ANSWER                                                                 |
+------------------------------------------------------------------------+
| The answer is 7am.                                                     |
+------------------------------------------------------------------------+



---
## Part 6: Watching it choose badly

Our twin picks its tool by **spotting keywords**. That is easy to understand and easy to break.
Knowing how it breaks is the point of this step.

In [6]:
#@title ▶ Step 6 — Try to make my twin pick the wrong tool { display-mode: "form" }
#@markdown Some questions that cause trouble — or type your own:
question = "How many days until my 10 km race?" #@param ["How many days until my 10 km race?", "What is the time value of 500 * 12?", "Is today a good day to run 5 km?", "What is my best 5km?"] {allow-input: true}

# ---- hidden helper code: learners never need to read this ----
import sys
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

import re
def show_text(label, text, width=76):
    if label: print(label)
    for line in str(text).splitlines():
        while len(line) > width:
            cut = line.rfind(" ", 0, width)
            if cut <= 0: cut = width
            print("   " + line[:cut]); line = line[cut:].lstrip()
        print("   " + line)
def box(title, text, width=72):
    import textwrap
    print("+" + "-" * width + "+")
    print("| " + title.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
    for para in str(text).split(chr(10)):
        if not para.strip():
            print("| " + " " * (width - 2) + " |"); continue
        for line in textwrap.wrap(para, width - 4) or [""]:
            print("| " + line.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
def bar(part, whole, width=34):
    if whole <= 0: return "[" + "." * width + "]"
    f = max(0, min(width, int(round(width * part / whole))))
    return "[" + "#" * f + "." * (width - f) + "]"
def pct(part, whole):
    return (format(100.0 * part / whole, ".0f") + "%") if whole else "-"

import ast, operator, datetime
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.Pow: operator.pow, ast.Mod: operator.mod,
        ast.USub: operator.neg, ast.UAdd: operator.pos}
def safe_calc(expression):
    """Work out a sum WITHOUT python's eval, which would run any code it was given."""
    def ev(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
            return _OPS[type(node.op)](ev(node.left), ev(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
            return _OPS[type(node.op)](ev(node.operand))
        raise ValueError("only numbers and + - * / % ** are allowed")
    return ev(ast.parse(str(expression).strip(), mode="eval").body)
def tool_calculator(expression):
    try:
        return {"result": safe_calc(expression), "expression": expression}
    except Exception as e:
        return {"error": str(e)}
def tool_clock(_=None):
    now = datetime.datetime.now()
    return {"time": now.strftime("%H:%M"), "date": now.strftime("%Y-%m-%d"),
            "day": now.strftime("%A")}
RATES = {"km->miles": 0.621371, "miles->km": 1.60934, "kg->pounds": 2.20462,
         "pounds->kg": 0.453592, "celsius->fahrenheit": None}
def tool_convert(request):
    m = re.search(r"(-?\d+(?:\.\d+)?)\s*([a-zA-Z]+)", str(request))
    if not m: return {"error": "I could not find a number and a unit in that"}
    amount = float(m.group(1)); unit = m.group(2).lower()
    table = {"km": ("miles", 0.621371), "miles": ("km", 1.60934), "mile": ("km", 1.60934),
             "kg": ("pounds", 2.20462), "pounds": ("kg", 0.453592), "lb": ("kg", 0.453592)}
    if unit not in table: return {"error": "I only know km, miles, kg and pounds - not '" + unit + "'"}
    to, factor = table[unit]
    return {"result": round(amount * factor, 2), "from": str(amount) + " " + unit, "to_unit": to}
def my_facts():
    return globals().get("_FACTS", {})
def _norm(word):
    return word.lower().strip("?.,!'").rstrip("s")
def _words(text):
    return [_norm(w) for w in re.findall(r"[A-Za-z0-9]+", str(text))]
def fact_match(question):
    """Match on words, so 'pharmacy opens' still matches 'does my pharmacy open?'."""
    qw = set(_words(question))
    best, best_len = None, 0
    for key in my_facts():
        kw = [w for w in _words(key) if w]
        if kw and all(w in qw for w in kw) and len(kw) > best_len:
            best, best_len = key, len(kw)
    return best
def tool_facts(question):
    key = fact_match(question)
    if key is None:
        return {"error": "nothing in my fact list matches that"}
    return {"result": my_facts()[key], "matched": key}
TOOLBOX = {
 "calculator": (tool_calculator, "work out a sum"),
 "clock":      (tool_clock,      "say what the time, date or day is"),
 "converter":  (tool_convert,    "convert km, miles, kg and pounds"),
 "my facts":   (tool_facts,      "look something up in my own list of facts"),
}
def enabled_tools():
    return globals().get("_ENABLED", ["calculator", "clock"])
def think(question):
    """Decide which tool to reach for. Keyword matching - deliberately simple."""
    q = str(question).lower()
    on = enabled_tools()
    if "my facts" in on and fact_match(question):
        return "my facts", "this matches '" + fact_match(question) + "' in my own fact list"
    if "converter" in on and re.search(r"\d", q) and any(u in q for u in
            ("km", "mile", "kg", "pound", "lb", "convert")):
        return "converter", "there is a number next to a unit of distance or weight"
    if "clock" in on and any(w in q for w in ("time", "date", "day", "today")):
        return "clock", "the question mentions the time, the date or the day"
    if "calculator" in on and (re.search(r"\d\s*[-+*/%]\s*\d", q)
                               or any(w in q for w in ("calculate", "times", "plus", "minus"))):
        return "calculator", "there is a sum in the question"
    return None, "nothing matched, so I will answer from memory"
def extract_sum(question):
    m = re.search(r"[-+]?[\d.]+(?:\s*[-+*/%]\s*[-+]?[\d.]+)+", str(question))
    return m.group(0).strip() if m else None
def answer_from(tool_name, observation):
    if "error" in observation:
        return "I tried my " + tool_name + " but it said: " + observation["error"]
    if tool_name == "clock":
        return "It is " + observation["time"] + " on " + observation["day"] + ", " + observation["date"] + "."
    if tool_name == "converter":
        return observation["from"] + " is " + str(observation["result"]) + " " + observation["to_unit"] + "."
    return "The answer is " + str(observation["result"]) + "."
def run_agent(question, verbose=True):
    """Thought -> Action -> Observation -> Answer."""
    tool_name, why = think(question)
    if verbose:
        print("THOUGHT      : " + why)
    if tool_name is None:
        if verbose:
            print("ACTION       : none - answering from memory")
            print("OBSERVATION  : (nothing was looked up)")
        return None, "I do not have a tool for that, so I would have to guess."
    arg = extract_sum(question) if tool_name == "calculator" else question
    if tool_name == "calculator" and arg is None:
        arg = question
    if verbose:
        print("ACTION       : use the " + tool_name + " with  " + str(arg)[:50])
    observation = TOOLBOX[tool_name][0](arg)
    if verbose:
        print("OBSERVATION  : " + str(observation))
    return tool_name, answer_from(tool_name, observation)

globals()["_ENABLED"] = ["calculator", "clock", "converter", "my facts"]
globals()["_FACTS"] = {"best 5km": "23 minutes", "race": "14 March"}
print("QUESTION     : " + question)
print("")
used, reply = run_agent(question)
print("")
box("ANSWER", reply)
print("")
if used:
    note("It reached for the " + used + " tool. Is that what YOU would have picked?")
print("")
print("Why this happens: the twin matched a KEYWORD, not the meaning. 'days' looks")
print("like a clock question. '10 km' looks like a conversion. It never understood")
print("that you wanted a date subtracted.")
print("")
problem("A keyword router is confidently wrong at the edges - and the answer still reads fine.")
print("")
print("Real agents hand this decision to the model itself, describing each tool and")
print("letting it choose. Better, but not perfect: it still picks wrong sometimes,")
print("which is why Week 3.2 is about testing.")

QUESTION     : How many days until my 10 km race?

THOUGHT      : this matches 'race' in my own fact list
ACTION       : use the my facts with  How many days until my 10 km race?
OBSERVATION  : {'result': '14 March', 'matched': 'race'}

+------------------------------------------------------------------------+
| ANSWER                                                                 |
+------------------------------------------------------------------------+
| The answer is 14 March.                                                |
+------------------------------------------------------------------------+




Why this happens: the twin matched a KEYWORD, not the meaning. 'days' looks
like a clock question. '10 km' looks like a conversion. It never understood
that you wanted a date subtracted.




Real agents hand this decision to the model itself, describing each tool and
letting it choose. Better, but not perfect: it still picks wrong sometimes,
which is why Week 3.2 is about testing.


---
## What you just did

| Step | In plain words | The real name for it |
|---|---|---|
| 1 | Saw a model guess instead of check | Hallucination |
| 2 | Handed one job to real software | A tool |
| 3 | Listed what the twin may use | Tool registry |
| 4 | Thought → action → observation → answer | The ReAct loop |
| 5 | Added facts only you know | A custom tool |
| 6 | Watched it pick the wrong tool | Tool-selection failure |

### The one sentence to remember
**A tool turns a guess into a lookup.** Everything else — registries, loops, agents — is
plumbing around that single idea.

### One safety point worth keeping
Our calculator accepts numbers and arithmetic only, and refuses everything else. A tool that
ran whatever text it was handed would let anyone who can type to your twin run code on your
machine. **Tools are doors into your system — decide deliberately how wide each one opens.**

### Where this is going
Your twin can now act. **Week 3.2** asks whether it acts *correctly*, by writing tests for it.

### Want to see the code?
Click **Show code** on any step.